# Harmonia — Preprocessing Pipeline

Runs three steps in sequence:
1. Download + filter the Lakh MIDI dataset (~176k files → ~50k)
2. Tokenize with miditok REMI and build training examples
3. Push the dataset to HuggingFace Hub

**Runtime:** CPU is fine. No GPU needed here.
**Time estimate:** ~1-2 hours total depending on Colab CPU speed.

Outputs are saved to Google Drive so nothing is lost if the session drops.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/sannfh/harmonia.git /content/harmonia
%cd /content/harmonia
!pip install -e "." -q
!pip install datasets -q

In [ ]:
# ---- FILL THESE IN BEFORE RUNNING ----
HF_TOKEN = "hf_..."                          # Your HuggingFace write token
HF_DATASET_REPO = "your-username/harmonia-midi"  # Where to push the dataset
# ---------------------------------------

DRIVE_DIR = "/content/drive/MyDrive/harmonia"
DATA_DIR  = f"{DRIVE_DIR}/data"

!mkdir -p {DATA_DIR}

## Step 1 — Download and filter the Lakh MIDI dataset

Downloads `lmd_full.tar.gz` (~1.5 GB) and applies quality filters in parallel.
Writes a text file of passing file paths to Drive.

**Expected output:** ~50k–70k files passing out of ~176k.

In [ ]:
# Extract and filter on local disk (fast SSD), outputs saved to Drive
# If the archive is already on Drive from a previous run, --archive points to it
# so it won't re-download — it just extracts locally and filters.

LOCAL_LAKH   = "/content/lakh"
DRIVE_ARCHIVE = f"{DATA_DIR}/lakh/lmd_full.tar.gz"

!python preprocessing/filter_lakh.py \
    --data-dir {LOCAL_LAKH} \
    --archive  {DRIVE_ARCHIVE} \
    --output   {DATA_DIR}/filtered.txt \
    --workers  2

In [ ]:
# Sanity check — print first 5 passing paths and total count
with open(f"{DATA_DIR}/filtered.txt") as f:
    lines = f.read().splitlines()
print(f"Total passing: {len(lines):,}")
for line in lines[:5]:
    print(" ", line)

## Step 2 — Tokenize filtered files

For each file: detects key, extracts instruments and tempo, builds a conditioning
prompt, then tokenizes with miditok REMI. Saves one training example per line
as a JSONL file.

**Expected output:** JSONL with ~40k–60k examples.

In [ ]:
!python preprocessing/tokenize_dataset.py \
    --file-list {DATA_DIR}/filtered.txt \
    --output    {DATA_DIR}/train.jsonl \
    --workers   2

In [ ]:
# Sanity check — print the first training example
import json

with open(f"{DATA_DIR}/train.jsonl") as f:
    first = json.loads(f.readline())

# Show full conditioning prompt + first 300 chars of MIDI tokens
print(first["text"][:600])
print("...")

## Step 3 — Push dataset to HuggingFace Hub

Loads the JSONL and pushes it as a private HuggingFace dataset.
The training notebook will stream it from there.

In [ ]:
!python preprocessing/create_hf_dataset.py \
    --input    {DATA_DIR}/train.jsonl \
    --repo-id  {HF_DATASET_REPO} \
    --hf-token {HF_TOKEN}

In [ ]:
# Confirm it's on the Hub
from datasets import load_dataset

ds = load_dataset(HF_DATASET_REPO, token=HF_TOKEN, split="train")
print(f"Dataset on Hub: {len(ds):,} examples")
print(ds[0]["text"][:300])